## Insurance charges — rubric walkthrough

This notebook covers: loading and previewing the data; SweetViz EDA; defining **X** / **y**, train/test split (80/20); scaling features; linear regression vs random forest baselines with **MSE** and **R²** on the test set; and **GridSearchCV** over `n_estimators`, `max_depth`, `min_samples_split`, and `max_features`.


In [22]:
# import libraries 
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression 



In [23]:
df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [24]:
# visualize the data 
report = sv.analyze(df)
report.show_html()

                                             |          | [  0%]   00:00 -> (? left)

Report SWEETVIZ_REPORT.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### EDA — main trends (from SweetViz)

- **`charges`** is **right-skewed**: most policies are relatively cheap, but a minority have very large costs (long upper tail).
- **Smoking** is the clearest driver: smokers sit much higher in the **charges** distribution than non-smokers.
- **Age** and **BMI** both move together with **charges** in a broad way: older and higher-BMI individuals tend to have higher average costs.
- **Number of children**, **sex**, and **region** matter less than smoking; any regional or sex differences are modest compared to smoking status.


In [25]:
# encode categorical variables

pd_encoded = pd.get_dummies(df, drop_first=True).astype(int)
pd_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [26]:
# preparing the data for modeling (use encoded data so all columns are numeric)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

y = pd_encoded["charges"]
X = pd_encoded.drop(columns=["charges"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# baseline linear regression model
baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train)
y_pred_lr = baseline_model.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# baseline random forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Linear Regression: MSE = {mse_lr:.2f}, R2 = {r2_lr:.4f}")
print(f"Random Forest:     MSE = {mse_rf:.2f}, R2 = {r2_rf:.4f}")


Linear Regression: MSE = 33566439.74, R2 = 0.7838
Random Forest:     MSE = 22118860.12, R2 = 0.8575


### Baseline models — performance on **test** data

- **Linear regression** is a simple baseline: it usually gets a decent **R²** but can miss nonlinear interactions (e.g., smoking with age/BMI), so **MSE** is often **higher** than tree models.
- **Random forest** can fit nonlinearities and interactions without manual feature engineering, so it often earns a **lower test MSE** and **higher R²** than linear regression on this dataset.
- The printed **MSE** and **R²** above are computed on the **held-out 20% test set** after fitting on the scaled training data.


In [27]:
#cross validation 
lr_scores = cross_val_score(baseline_model, X_train_scaled, y_train, cv=5, scoring="neg_mean_squared_error")
rf_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="neg_mean_squared_error")

print(f"Linear Regression: MSE = {-lr_scores.mean():.2f} (+/- {lr_scores.std() * 2:.2f})")
print(f"Random Forest:     MSE = {-rf_scores.mean():.2f} (+/- {rf_scores.std() * 2:.2f})")
# cv r2 scores 
lr_r2_scores = cross_val_score(baseline_model, X_train_scaled, y_train, cv=5, scoring="r2")
rf_r2_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="r2")

print(f"Linear Regression: R2 = {lr_r2_scores.mean():.4f} (+/- {lr_r2_scores.std() * 2:.4f})")
print(f"Random Forest:     R2 = {rf_r2_scores.mean():.4f} (+/- {rf_r2_scores.std() * 2:.4f})")



Linear Regression: MSE = 37953893.30 (+/- 10132046.57)
Random Forest:     MSE = 24997437.64 (+/- 8850651.10)
Linear Regression: R2 = 0.7331 (+/- 0.0982)
Random Forest:     R2 = 0.8232 (+/- 0.0815)


In [28]:
# grid search — use a fresh estimator (not the fitted rf_model)
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "max_features": [None, "sqrt", 0.6],
}

# n_jobs=1 on the forest avoids oversubscribing CPUs when GridSearchCV runs parallel folds
rf_for_grid = RandomForestRegressor(random_state=42, n_jobs=1)
grid_search = GridSearchCV(
    estimator=rf_for_grid,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_scaled, y_train)

# scoring uses *negative* MSE (sklearn maximizes score), so flip the sign for real MSE
best_mse = -grid_search.best_score_
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV mean MSE: {best_mse:,.2f}")


Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'max_depth': 10, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 300}
Best CV mean MSE: 21,815,581.88


### GridSearchCV — best hyperparameters

The search tunes **`n_estimators`**, **`max_depth`**, **`min_samples_split`**, and **`max_features`**. After the grid cell runs, the **best combination** is the `Best parameters: { ... }` line printed above (`grid_search.best_params_`). That set minimizes **mean cross-validated MSE** (scoring uses `neg_mean_squared_error`; **Best CV mean MSE** converts back to ordinary MSE).


In [30]:
# top 20 by CV score — sort by the *same* index so params match their scores
import numpy as np

scores = grid_search.cv_results_["mean_test_score"]
rank = np.argsort(-scores)[:20]
for i in rank:
    mse = -scores[i]
    print(f"MSE={mse:,.2f} | {grid_search.cv_results_['params'][i]}")


MSE=21,815,581.88 | {'max_depth': 10, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 300}
MSE=21,861,048.16 | {'max_depth': 10, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 200}
MSE=21,904,376.94 | {'max_depth': 10, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 100}
MSE=21,979,376.85 | {'max_depth': 20, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 300}
MSE=21,979,376.85 | {'max_depth': None, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 300}
MSE=21,979,376.85 | {'max_depth': 30, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 300}
MSE=22,048,143.38 | {'max_depth': None, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 200}
MSE=22,048,143.38 | {'max_depth': 20, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 200}
MSE=22,048,143.38 | {'max_depth': 30, 'max_features': 0.6, 'min_samples_split': 10, 'n_estimators': 200}
MSE=22,075,613.78 | {'max_depth': 30, 'max_features

I’d use n_estimators=300, max_depth=10, min_samples_split=10, max_features=0.6

Why: more trees (300) averaged better in my grid; max_depth=10 worked better than really deep trees; min_samples_split=10 forced splits to use a bit more data so the model didn’t chase noise; max_features=0.6 used a random subset of features each split, which seemed to generalize better than using every feature every time.

I compared the tuned random forest (GridSearchCV’s best_estimator_, refit on all of training) on training vs test using MSE and R².

Training: MSE ≈ 12.5M, R² ≈ 0.913
Test: MSE ≈ 20.1M, R² ≈ 0.871
Gap: train R² is about 0.04 higher than test R²; train MSE is lower than test MSE.
Is it overfitting? A little, but not badly. Test performance is worse than train performance, which is normal: the model is fit on the training set, so it will usually do better there. Overfitting would mean a big gap—e.g. train R² very high and test R² much lower, or train error much smaller than test error. Here the R² gap is small and test R² is still strong (~0.87), so the model generalizes reasonably well. I know that because I’m looking at the same metrics (MSE and R²) on train vs the held-out test set I never fit on.